# Bronze Layer - Ingestao dos Dados Brutos

**Camada:** Bronze (raw)
**Objetivo:** carregar os arquivos de origem para o Delta Lake **exatamente como vieram**.

Regras desta camada:
- Le os arquivos da pasta `source/` (enviada ao workspace).
- Mantem TODAS as colunas como STRING (sem interpretacao de tipos).
- Nao faz limpeza, validacao ou transformacao de dados (isso fica na Silver).
- Unica adaptacao tecnica: normaliza os NOMES das colunas (o Delta nao aceita
  espacos/acentos). O conteudo dos dados nao e alterado.

Tabelas geradas em `workspace.bronze.*` (8 no total):
- 5 agregados da Base dos Dados: `indicador_alfabetizacao_uf`, `indicador_alfabetizacao_municipio`,
  `meta_alfabetizacao_brasil`, `meta_alfabetizacao_uf`, `meta_alfabetizacao_municipio`
- 3 microdados do INEP: `ts_aluno`, `ts_municipio`, `ts_estado` (uniao dos anos 2023-2025)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import re

spark = SparkSession.builder.appName("tech-challenge-bronze").getOrCreate()

CATALOG = "workspace"
SCHEMA = "bronze"

# Pasta (workspace/DBFS) onde os arquivos de source/ foram enviados.
# Gerada localmente por: python projeto/bronze/prep_source.py  (depois faca o upload)
SOURCE_PATH = "/Workspace/Users/justinofilipe03@gmail.com/TechChallenge_2/source"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema {CATALOG}.{SCHEMA} pronto")

In [0]:
def normalizar_nome_coluna(nome):
    """Normaliza nomes de coluna (Delta nao aceita espacos/acentos).

    NAO e tratamento de dados - so uma limitacao tecnica do formato.
    O conteudo das colunas continua exatamente como veio.
    """
    import unicodedata
    nome = unicodedata.normalize("NFKD", nome).encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome).strip("_")
    return nome


def _normalizar_colunas(df):
    for c in df.columns:
        novo = normalizar_nome_coluna(c)
        if novo != c:
            df = df.withColumnRenamed(c, novo)
    return df

In [0]:
def ingerir_agregado(nome_arquivo, nome_tabela):
    """Ingesta um CSV agregado da Base dos Dados (separador ',' , UTF-8)."""
    caminho = f"{SOURCE_PATH}/{nome_arquivo}"
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "false")
          .option("sep", ",")
          .option("encoding", "UTF-8")
          .load(caminho))
    df = _normalizar_colunas(df)
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{nome_tabela}"))
    print(f"{nome_tabela}: {df.count():,} registros")
    return df


def ingerir_microdado(subpasta, nome_tabela):
    """Ingesta um microdado do INEP unindo os anos (arquivos AAAA.csv).

    Os microdados usam ';'. Como o schema pode
    mudar entre anos (ex.: ts_municipio 2023 x 2024), lemos arquivo a arquivo e
    unimos por NOME de coluna. A coluna 'ano' e derivada do nome do arquivo.
    """
    base = f"{SOURCE_PATH}/{subpasta}"
    arquivos = [f.path for f in dbutils.fs.ls(base) if f.name.endswith(".csv")]  # noqa: F821
    df = None
    for arq in arquivos:
        ano = re.search(r"(20\d{2})", arq).group(1)
        parte = (spark.read.format("csv")
                 .option("header", "true")
                 .option("inferSchema", "false")
                 .option("sep", ";")
                 .option("encoding", "ISO-8859-1")
                 .load(arq)
                 .withColumn("ano", F.lit(ano)))
        df = parte if df is None else df.unionByName(parte, allowMissingColumns=True)
    df = _normalizar_colunas(df)
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .partitionBy("ano")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{nome_tabela}"))
    print(f"{nome_tabela}: {df.count():,} registros")
    return df

## Ingestao dos agregados (Base dos Dados)

In [0]:
ingerir_agregado("indicador_alfabetizacao_uf.csv.gz", "indicador_alfabetizacao_uf")
ingerir_agregado("indicador_alfabetizacao_municipio.csv.gz", "indicador_alfabetizacao_municipio")
ingerir_agregado("meta_alfabetizacao_brasil.csv.gz", "meta_alfabetizacao_brasil")
ingerir_agregado("meta_alfabetizacao_uf.csv.gz", "meta_alfabetizacao_uf")
ingerir_agregado("meta_alfabetizacao_municipio.csv.gz", "meta_alfabetizacao_municipio")

## Ingestao dos microdados (INEP) - uniao dos anos 2023-2025

In [0]:
ingerir_microdado("ts_aluno", "ts_aluno")
ingerir_microdado("ts_municipio", "ts_municipio")
ingerir_microdado("ts_estado", "ts_estado")

## Validacao final

In [0]:
tabelas = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
print(f"Tabelas bronze criadas ({len(tabelas)}):")
for t in tabelas:
    c = spark.table(f"{CATALOG}.{SCHEMA}.{t.tableName}").count()
    print(f"  {t.tableName}: {c:,} registros")